In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SPLIT_DIR = PROJECT_ROOT / "experiments"

split_manifest = pd.read_csv(
    SPLIT_DIR / "office31_splits.csv"
)

source_train = split_manifest[
    split_manifest["split"] == "source_train"
].reset_index(drop=True)

source_test = split_manifest[
    split_manifest["split"] == "source_test"
].reset_index(drop=True)

target_adapt_paths = split_manifest.loc[
    split_manifest["split"] == "target_adapt",
    "path"
].tolist()

target_test = split_manifest[
    split_manifest["split"] == "target_test"
].reset_index(drop=True)

In [2]:
print("Source train:", len(source_train))
print("Source test:", len(source_test))
print("Target adaptation:", len(target_adapt_paths))
print("Target test:", len(target_test))

Source train: 2253
Source test: 564
Target adaptation: 397
Target test: 398


### Feature Extraction

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from PIL import Image
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
NUM_WORKERS = 0

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
# Load pretrained ImageNet weights
weights = models.ResNet18_Weights.DEFAULT

# Image preprocessing associated with these weights
preprocess = weights.transforms()

# Load ResNet-18
model = models.resnet18(weights=weights)

# Remove the original ImageNet classifier
model.fc = nn.Identity()

# Freeze the feature extractor
for param in model.parameters():
    param.requires_grad = False

# Move to GPU
model = model.to(DEVICE)

# Set evaluation mode
model.eval()

print("ResNet-18 loaded successfully!")
print("Final layer:", model.fc)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\lenovo/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:06<00:00, 7.43MB/s]


ResNet-18 loaded successfully!
Final layer: Identity()


In [5]:
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)

with torch.no_grad():
    output = model(dummy_input)

print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 512])


In [6]:
class ImageFeatureDataset(Dataset):

    def __init__(self, paths, labels=None, transform=None):
        self.paths = list(paths)
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        image_path = self.paths[idx]

        with Image.open(image_path) as img:
            image = img.convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.labels is not None:
            return image, self.labels[idx]

        return image

In [7]:
def make_loader(paths, labels=None):

    dataset = ImageFeatureDataset(
        paths=paths,
        labels=labels,
        transform=preprocess
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

In [9]:
CLASSES = sorted(source_train["label"].unique())

class_to_idx = {
    name: idx for idx, name in enumerate(CLASSES)
}

In [10]:
source_train_loader = make_loader(
    source_train["path"],
    source_train["label"].map(class_to_idx).tolist()
)

source_test_loader = make_loader(
    source_test["path"],
    source_test["label"].map(class_to_idx).tolist()
)

# No labels for target adaptation data
target_adapt_loader = make_loader(
    target_adapt_paths
)

target_test_loader = make_loader(
    target_test["path"],
    target_test["label"].map(class_to_idx).tolist()
)

In [11]:
def extract_features(model, dataloader):

    all_features = []

    model.eval()

    with torch.no_grad():

        for batch in dataloader:

            # Separate images from labels, if labels exist
            if isinstance(batch, (tuple, list)):
                images = batch[0]
            else:
                images = batch

            # Move images to GPU
            images = images.to(DEVICE)

            # Forward propagation through ResNet
            features = model(images)

            # Move features back to CPU
            all_features.append(features.cpu())

    return torch.cat(all_features, dim=0)

In [12]:
Xs_train = extract_features(model, source_train_loader)

Xs_test = extract_features(model, source_test_loader)

Xt_adapt = extract_features(model, target_adapt_loader)

Xt_test = extract_features(model, target_test_loader)

In [13]:
print("Amazon source train:", Xs_train.shape)
print("Amazon source test:", Xs_test.shape)

print("Webcam target adaptation:", Xt_adapt.shape)
print("Webcam target test:", Xt_test.shape)

Amazon source train: torch.Size([2253, 512])
Amazon source test: torch.Size([564, 512])
Webcam target adaptation: torch.Size([397, 512])
Webcam target test: torch.Size([398, 512])


In [14]:
for name, features in {
    "Xs_train": Xs_train,
    "Xs_test": Xs_test,
    "Xt_adapt": Xt_adapt,
    "Xt_test": Xt_test
}.items():

    print(
        name,
        "Finite:", torch.isfinite(features).all().item(),
        "Mean:", features.mean().item(),
        "Std:", features.std().item()
    )

Xs_train Finite: True Mean: 0.8064960837364197 Std: 0.9086400866508484
Xs_test Finite: True Mean: 0.802026093006134 Std: 0.9091354012489319
Xt_adapt Finite: True Mean: 0.7827379703521729 Std: 0.8286232352256775
Xt_test Finite: True Mean: 0.7858527898788452 Std: 0.8290168046951294


In [16]:
SEED = 42

In [17]:
FEATURE_DIR = PROJECT_ROOT / "experiments" / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

feature_file = FEATURE_DIR / "office31_amazon_webcam_features.pt"

torch.save(
    {
        "Xs_train": Xs_train,
        "Xs_test": Xs_test,
        "Xt_adapt": Xt_adapt,
        "Xt_test": Xt_test,
        "ys_train": torch.tensor(
            source_train["label"].map(class_to_idx).tolist()
        ),
        "ys_test": torch.tensor(
            source_test["label"].map(class_to_idx).tolist()
        ),
        "yt_test": torch.tensor(
            target_test["label"].map(class_to_idx).tolist()
        ),
        "classes": CLASSES,
        "class_to_idx": class_to_idx,
        "seed": SEED
    },
    feature_file
)

print("Saved to:", feature_file)

Saved to: d:\DUK- Research Collab\Sanjay PhD\cnn-domain-adaptation-optimal-transport\experiments\features\office31_amazon_webcam_features.pt


In [18]:
print("File exists:", feature_file.exists())
print("File size (MB):", round(feature_file.stat().st_size / (1024**2), 2))

File exists: True
File size (MB): 7.08
